# SF Parcel Market Value Estimation

This notebook estimates present market values for San Francisco residential properties by:
1. Training on recently sold parcels (where assessed value ≈ market value)
2. Applying predictions to properties without recent sales (where Prop 13 caps assessed values)

**Data Source**: Assessor Historical Secured Property Tax Rolls (2024 closed roll year)

## Section 1: Setup and Data Fetch

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
input_path = Path('input.csv')

if input_path.exists():
    print(f"Loading from {input_path}")
    df = pd.read_csv(input_path)
else:
    url = "https://data.sfgov.org/resource/wv5m-vpq2.csv?$limit=250000&$where=closed_roll_year='2024'"
    print(f"Fetching from {url}")
    df = pd.read_csv(url)

print(f"Loaded {len(df):,} records")

In [3]:
numeric_cols = [
    'number_of_bedrooms', 'number_of_bathrooms', 'number_of_rooms',
    'number_of_stories', 'number_of_units', 'lot_depth', 'lot_frontage',
    'lot_area', 'property_area', 'basement_area', 'year_property_built',
    'assessed_land_value', 'assessed_improvement_value', 'percent_of_ownership'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['current_sales_date'] = pd.to_datetime(df['current_sales_date'], errors='coerce')

df = df.rename(columns={
    'current_sales_date': 'sales_date',
    'number_of_bedrooms': 'bedrooms',
    'number_of_bathrooms': 'bathrooms',
    'number_of_rooms': 'rooms',
    'number_of_stories': 'stories',
    'number_of_units': 'units',
    'year_property_built': 'year_built',
    'assessor_neighborhood': 'neighborhood',
    'lot_area': 'lot_area_filled',
    'percent_of_ownership': 'ownership_pct'
})

df['total_assessed_value'] = df['assessed_land_value'].fillna(0) + df['assessed_improvement_value'].fillna(0)

## Section 2: Training Data Preparation

In [4]:
cutoff_date = pd.Timestamp('2021-02-08')
recent_sales = df[df['sales_date'] >= cutoff_date].copy()
print(f"Sales after Feb 8, 2021: {len(recent_sales):,}")

Sales after Feb 8, 2021: 23,602


In [5]:
cutoff_2024 = pd.Timestamp('2024-01-01')
recent_sales = recent_sales[recent_sales['sales_date'] < cutoff_2024]
print(f"After excluding 2024+ sales: {len(recent_sales):,}")

After excluding 2024+ sales: 19,659


In [6]:
recent_sales = recent_sales[recent_sales['total_assessed_value'] > 0]
print(f"After excluding zero-value parcels: {len(recent_sales):,}")

After excluding zero-value parcels: 19,571


In [7]:
residential_patterns = [
    'single family',
    'condominium',
    'two family',
    'apartments',
    'three family',
    'tenancies in common',
    'four family',
    'condo',
    'multiple family',
    'multi-family',
    'dwelling',
    'cooperative',
    'live/work',
    'residential'
]

def is_residential(use_def):
    if pd.isna(use_def):
        return False
    use_lower = use_def.lower()
    return any(pattern in use_lower for pattern in residential_patterns)

print(f"Use definitions in recent_sales:")
print(recent_sales['use_definition'].value_counts().head(20))

recent_sales = recent_sales[recent_sales['use_definition'].apply(is_residential)]
print(f"\nAfter filtering to residential: {len(recent_sales):,}")

Use definitions in recent_sales:
use_definition
Single Family Residential    16345
Multi-Family Residential      2307
Commercial Misc                252
Commercial Retail              236
Miscellaneous/Mixed-Use        182
Industrial                     121
Commercial Office               81
Commercial Hotel                47
Name: count, dtype: int64

After filtering to residential: 18,652


In [8]:
training_data = recent_sales.copy()
print(f"\nFinal training set: {len(training_data):,} parcels")


Final training set: 18,652 parcels


## Section 3: Feature Engineering

In [9]:
def clean_property_type(use_def):
    if pd.isna(use_def):
        return 'Other'
    use_lower = use_def.lower()
    
    if 'single family' in use_lower:
        return 'Single Family'
    elif any(x in use_lower for x in ['condominium', 'condo', 'tenancies in common', 'cooperative', 'tic']):
        return 'Condo/TIC'
    elif any(x in use_lower for x in ['two family', 'three family', 'four family', 'multi-family', 'multiple family', 'apartments']):
        return 'Multi-Family'
    elif any(x in use_lower for x in ['mixed', 'live/work', 'commercial']):
        return 'Mixed Use'
    elif 'residential' in use_lower:
        return 'Single Family'
    return 'Other'

def consolidate_categories(series, min_count=30):
    counts = series.value_counts()
    valid_cats = counts[counts >= min_count].index
    return series.apply(lambda x: x if x in valid_cats else 'Other')

In [10]:
def build_imputation_lookup(data):
    data = data.copy()
    data['property_type'] = data['use_definition'].apply(clean_property_type)
    data['sqft_bucket'] = (data['property_area'] // 250) * 250
    
    lookup = data.groupby(['property_type', 'sqft_bucket']).agg({
        'bedrooms': 'median',
        'bathrooms': 'median',
        'rooms': 'median'
    }).reset_index()
    
    global_medians = {
        'bedrooms': data['bedrooms'].median(),
        'bathrooms': data['bathrooms'].median(),
        'rooms': data['rooms'].median()
    }
    
    return lookup, global_medians

lookup_df, global_medians = build_imputation_lookup(training_data)

In [11]:
def engineer_features(data, lookup_df, global_medians):
    data = data.copy()
    
    data['use_definition_clean'] = data['use_definition'].apply(clean_property_type)
    data['neighborhood_clean'] = consolidate_categories(data['neighborhood'].fillna('Unknown'), min_count=50)
    data['zoning_code_clean'] = consolidate_categories(data['zoning_code'].fillna('Unknown'), min_count=30)
    data['construction_type_clean'] = consolidate_categories(data['construction_type'].fillna('Unknown'), min_count=30)
    
    data['bedrooms_missing'] = data['bedrooms'].isna().astype(int)
    data['bathrooms_missing'] = data['bathrooms'].isna().astype(int)
    data['rooms_missing'] = data['rooms'].isna().astype(int)
    
    data['sqft_bucket'] = (data['property_area'] // 250) * 250
    
    for col in ['bedrooms', 'bathrooms', 'rooms']:
        merged = data.merge(
            lookup_df[['property_type', 'sqft_bucket', col]],
            left_on=['use_definition_clean', 'sqft_bucket'],
            right_on=['property_type', 'sqft_bucket'],
            how='left',
            suffixes=('', '_lookup')
        )
        lookup_col = f'{col}_lookup'
        if lookup_col in merged.columns:
            data[col] = data[col].fillna(merged[lookup_col])
        data[col] = data[col].fillna(global_medians[col])
    
    data['building_age'] = 2024 - data['year_built'].fillna(1950)
    data['building_age'] = data['building_age'].clip(0, 200)
    
    data['lot_area_filled'] = data['lot_area_filled'].fillna(data['lot_area_filled'].median())
    data['property_area'] = data['property_area'].fillna(data['property_area'].median())
    data['basement_area'] = data['basement_area'].fillna(0)
    data['lot_depth'] = data['lot_depth'].fillna(data['lot_depth'].median())
    data['lot_frontage'] = data['lot_frontage'].fillna(data['lot_frontage'].median())
    data['stories'] = data['stories'].fillna(data['stories'].median())
    data['units'] = data['units'].fillna(1)
    data['ownership_pct'] = data['ownership_pct'].fillna(100)
    
    data['units_per_lot_sqft'] = data['units'] / data['lot_area_filled'].clip(lower=100)
    data['rooms_per_unit'] = data['rooms'] / data['units'].clip(lower=1)
    
    return data

training_data = engineer_features(training_data, lookup_df, global_medians)

In [12]:
numeric_features = [
    'bedrooms', 'bathrooms', 'rooms', 'stories', 'units',
    'lot_depth', 'lot_frontage', 'lot_area_filled', 'property_area', 'basement_area',
    'building_age', 'ownership_pct',
    'units_per_lot_sqft', 'rooms_per_unit',
    'bedrooms_missing', 'bathrooms_missing', 'rooms_missing'
]

categorical_features = [
    'use_definition_clean', 'neighborhood_clean', 'zoning_code_clean', 'construction_type_clean'
]

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

Numeric features: 17
Categorical features: 4


## Section 4: Model Training

In [13]:
training_data['log_value'] = np.log(training_data['total_assessed_value'])

X = training_data[numeric_features + categorical_features]
y = training_data['log_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train):,}")
print(f"Test set: {len(X_test):,}")

Training set: 14,921
Test set: 3,731


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), categorical_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(
        n_estimators=500,
        max_depth=7,
        learning_rate=0.08,
        subsample=0.8,
        min_samples_leaf=10,
        random_state=42
    ))
])

model.fit(X_train, y_train)
print("Model trained successfully")

Model trained successfully


## Section 5: Evaluation

In [15]:
y_pred = model.predict(X_test)

r2_log = r2_score(y_test, y_pred)

y_test_actual = np.exp(y_test)
y_pred_actual = np.exp(y_pred)

pct_errors = np.abs(y_pred_actual - y_test_actual) / y_test_actual * 100
median_pct_error = np.median(pct_errors)

median_abs_error = np.median(np.abs(y_pred_actual - y_test_actual))

within_25 = np.mean(pct_errors <= 25) * 100
within_50 = np.mean(pct_errors <= 50) * 100

print(f"R² (log scale): {r2_log:.3f}")
print(f"Median % error: {median_pct_error:.1f}%")
print(f"Median absolute error: ${median_abs_error:,.0f}")
print(f"Within 25% accuracy: {within_25:.1f}%")
print(f"Within 50% accuracy: {within_50:.1f}%")

R² (log scale): 0.675
Median % error: 15.0%
Median absolute error: $213,336
Within 25% accuracy: 71.9%
Within 50% accuracy: 92.1%


## Section 6: Generate Predictions

In [16]:
all_residential = df[df['use_definition'].apply(is_residential)].copy()
all_residential = all_residential[all_residential['total_assessed_value'] > 0]
print(f"Total residential parcels: {len(all_residential):,}")

Total residential parcels: 191,090


In [17]:
all_residential = engineer_features(all_residential, lookup_df, global_medians)

In [18]:
recent_sale_mask = (
    (all_residential['sales_date'] >= cutoff_date) & 
    (all_residential['sales_date'] < cutoff_2024)
)

all_residential['market_value'] = np.nan
all_residential['value_source'] = ''

all_residential.loc[recent_sale_mask, 'market_value'] = all_residential.loc[recent_sale_mask, 'total_assessed_value']
all_residential.loc[recent_sale_mask, 'value_source'] = 'assessed'

print(f"Parcels with recent sales (using assessed): {recent_sale_mask.sum():,}")

Parcels with recent sales (using assessed): 18,652


In [19]:
needs_prediction = ~recent_sale_mask
X_predict = all_residential.loc[needs_prediction, numeric_features + categorical_features]

log_predictions = model.predict(X_predict)
predictions = np.exp(log_predictions)

all_residential.loc[needs_prediction, 'market_value'] = predictions
all_residential.loc[needs_prediction, 'value_source'] = 'predicted'

print(f"Parcels with predicted values: {needs_prediction.sum():,}")

Parcels with predicted values: 172,438


In [20]:
predicted_parcels = all_residential[all_residential['value_source'] == 'predicted']
ratio = predicted_parcels['market_value'].median() / predicted_parcels['total_assessed_value'].median()
print(f"\nProp 13 sanity check:")
print(f"Median predicted market value: ${predicted_parcels['market_value'].median():,.0f}")
print(f"Median Prop 13 assessed value: ${predicted_parcels['total_assessed_value'].median():,.0f}")
print(f"Ratio (should be ~1.9x): {ratio:.2f}x")


Prop 13 sanity check:
Median predicted market value: $1,478,674
Median Prop 13 assessed value: $710,913
Ratio (should be ~1.9x): 2.08x


## Section 7: Export

In [21]:
output_cols = [
    'parcel_number', 'block', 'lot', 'property_location', 'neighborhood',
    'use_definition', 'zoning_code', 'year_built',
    'bedrooms', 'bathrooms', 'rooms', 'stories', 'units',
    'property_area', 'lot_area_filled',
    'total_assessed_value', 'market_value', 'value_source', 'sales_date'
]

output_df = all_residential[output_cols].copy()
output_df = output_df.rename(columns={'lot_area_filled': 'lot_area'})

output_df.to_csv('output.csv', index=False)
print(f"Exported {len(output_df):,} parcels to output.csv")

Exported 191,090 parcels to output.csv


In [22]:
print("\n=== Summary ===")
print(f"Total parcels exported: {len(output_df):,}")
print(f"  - With assessed values (recent sales): {(output_df['value_source'] == 'assessed').sum():,}")
print(f"  - With predicted values: {(output_df['value_source'] == 'predicted').sum():,}")
print(f"\nTotal market value: ${output_df['market_value'].sum():,.0f}")


=== Summary ===
Total parcels exported: 191,090
  - With assessed values (recent sales): 18,652
  - With predicted values: 172,438

Total market value: $347,183,729,344
